# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install numpy pandas matplotlib --upgrade pip --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import platform
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import shutil
import subprocess
import re
import xml.etree.ElementTree as ET
import sys

**Declarations**

Set the variables below and run the generation cell. 

- `scenario` is the path to the `.xml` scenario file that will be simulated.
- `legi_nr` will be put as seed for the `.xml` scenario.

In [ ]:
# --- Declarations: editable variables for students ---
scenario = ...
legi_nr = ...

### 1. Run a Scenario

#### 1.1. Update Local Repository

In [ ]:
# --- Jemula Class Paths ---
# jemula and jemula802 are one repository, cloned into PROJECT_DIR below.
# Every path here is built from it, so moving the checkout means editing one
# line.
PROJECT_DIR = "jemula-project"
REPO_URL = "https://gitlab.ethz.ch/wireless/jemula.git"
# The image Jemula802_Setup_Docker.ipynb builds, for the Docker route.
DOCKER_IMAGE = "jemula:latest"

lib = f"{PROJECT_DIR}/simulator/kernel/lib"
# webui holds the page every run writes next to its results (timeline.html).
common = f"./:{PROJECT_DIR}/simulator/engine/bin:{PROJECT_DIR}/simulator/kernel/bin:{PROJECT_DIR}/simulator/engine/webui"
classpath_jemula = f"{common}:{lib}/xstream-1.3.1.jar"
classpath_jemula802 = f"{classpath_jemula}:{lib}/commons-io-2.14.0.jar"

if platform.system() == "Windows":
    classpath_jemula   = classpath_jemula.replace(":", ";")
    classpath_jemula802 = classpath_jemula802.replace(":", ";")

In [ ]:
# --- Clone or Update the Jemula Repository ---
# One repository holding both jemula and jemula802, with the tests, the
# scenarios and the fixes.
if os.path.isdir(f"{PROJECT_DIR}/.git"):
    print(f"Updating {PROJECT_DIR} ...")
    if subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"]).returncode == 0:
        print(f"{PROJECT_DIR} updated.")
    else:
        # The update does not fit on top of this copy - usually because it holds
        # files you changed (a scenario's seed, an algorithm) that the update
        # changes too. Keep that copy, with your changes, and start afresh.
        import time
        kept = f"{PROJECT_DIR}-{time.strftime('%Y%m%d-%H%M%S')}"
        os.rename(PROJECT_DIR, kept)
        print(f"Could not update {PROJECT_DIR}. Your copy, with your changes, is kept as {kept}.")
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
        print(f"{PROJECT_DIR} cloned.")
else:
    print(f"Cloning {PROJECT_DIR} ...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    print(f"{PROJECT_DIR} cloned.")


#### 1.2 Update Scenario Seed

In [ ]:
# --- Update Scenario Seed from Matriculation Number (preserve formatting) ---
def update_seed_in_scenario_text(scenario_path, new_seed):
    with open(scenario_path, "r", encoding="utf-8") as f:
        content = f.read()

    # replace seed="..." only inside <JE802StatEval ...>
    new_content = re.sub(
        r'(<JE802StatEval[^>]*\bseed=")(\d+)(")',
        rf'\g<1>{new_seed}\3',
        content,
        count=1
    )

    if content != new_content:
        with open(scenario_path, "w", encoding="utf-8") as f:
            f.write(new_content)
        print(f"Updated seed to {new_seed} in {scenario_path}")
    else:
        print("No seed attribute found.")

update_seed_in_scenario_text(scenario, legi_nr)

#### 1.3. Start the Scenario

Jemula runs where you set it up: with your JDK after `Jemula802_Setup.ipynb`, or in Docker after `Jemula802_Setup_Docker.ipynb`. Your algorithms in `my-simulations/algorithms/` are compiled first, so edits take effect on the next run.

In [ ]:
# --- Run the scenario ---
# Jemula runs where you set it up: here with a JDK (Jemula802_Setup.ipynb), or
# in Docker (Jemula802_Setup_Docker.ipynb). Either way your algorithms in
# my-simulations/algorithms/ are compiled first, so the one you just edited is
# the one that runs, and the results land in results/ next to this notebook.
memory = "2G"   # Java heap for the run; raise it (e.g. "4G") for big scenarios
algorithms = f"{PROJECT_DIR}/my-simulations/algorithms/layer2_802Algorithms/*.java"

if os.path.isfile(f"{PROJECT_DIR}/simulator/engine/bin/emulator/JE802Starter.class"):
    compiled = subprocess.run(
        " ".join(["javac", "-g:none", "-nowarn", "-encoding", "UTF-8", "-classpath", classpath_jemula802,
                  "-d", f"{PROJECT_DIR}/simulator/engine/bin", algorithms]),
        shell=True, capture_output=True, text=True)
    if compiled.returncode != 0:
        print(compiled.stdout + compiled.stderr)
        raise SystemExit("Your algorithms do not compile - see the messages above. Nothing ran.")
    command = ["java", f"-Xms{memory}", f"-Xmx{memory}", "-classpath", classpath_jemula802,
               "emulator.JE802Starter", scenario]
elif shutil.which("docker") and subprocess.run(["docker", "images", "-q", DOCKER_IMAGE],
                                               capture_output=True, text=True).stdout.strip():
    lib = "/workspace/simulator/engine/build/install/engine/lib/*"
    command = ["docker", "run", "--rm", "-v", f"{os.getcwd()}:/work", "-w", "/work",
               "--entrypoint", "bash", DOCKER_IMAGE, "-lc",
               f"mkdir -p /tmp/alg && javac -nowarn -encoding UTF-8 -d /tmp/alg -cp '{lib}' {algorithms} "
               f"&& java -Xms{memory} -Xmx{memory} -cp '/tmp/alg:{lib}' emulator.JE802Starter '{scenario}'"]
else:
    raise SystemExit("Jemula is not set up yet: run Jemula802_Setup.ipynb (with a JDK) or "
                     "Jemula802_Setup_Docker.ipynb (with Docker) once, then this cell again.")

print(f"Running example scenario {scenario} ...")
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
process.wait()
print(f"\nExample scenario finished with exit code {process.returncode}")

### 2 Simulation Results

#### 2.1. Folder Location
We dynamically extract the folder location from the property `Path2Results` to see where the files are saved to.

In [ ]:
# --- Locate Results Folder ---
tree = ET.parse(scenario)
root = tree.getroot()
path_elem = root.find(".//JE802StatEval")
path2results = path_elem.get("Path2Results") if path_elem is not None else None

if not path2results:
    raise ValueError("Path2Results not found in scenario XML")

print(f"Found results folder: {path2results}")

# List contents (cross-platform)
if os.path.isdir(path2results):
    print(f"\nContents of {path2results}:")
    for f in os.listdir(path2results)[:10]:
        print("  - ", f)
    print("  -  ...")
else:
    print(f"\nResults folder {path2results} does not exist yet.")